# Explainable Phishing Email Scanner - Demo

This notebook demonstrates the end-to-end usage of the phishing email scanner, showcasing the ML-first approach with conditional LLM escalation.

In [ ]:
# Install dependencies if needed
# !pip install scikit-learn openai-agents joblib

In [ ]:
# Import required modules
import asyncio
import sys
sys.path.insert(0, '..')  # Add parent directory to path

from src.config import ScannerConfig
from src.pipeline.scanner import EmailScanner

## Architecture Overview

The phishing email scanner follows a modular pipeline:

1. **Feature Extraction**: Extracts named features from email text and sender
2. **ML Classification**: Uses LogisticRegression to produce risk score (0-100) and top signals
3. **Rule Engine**: Applies threshold-based classification (Safe: <45, Suspicious: 45-70, Phishing: >70)
4. **LLM Escalation**: Conditionally escalates borderline (45-70) or high-impact emails to LLM
5. **Explanation Generation**: Creates plain-English explanations referencing ML signals and optionally LLM insights

In [ ]:
# Initialize scanner with default configuration
config = ScannerConfig()
scanner = EmailScanner(config)

## 1. Scanning a Safe Email

In [ ]:
# Safe email example
safe_email = """Hi John,

Please find attached the quarterly report as discussed in our meeting yesterday.
The numbers look good and the team has been performing well.

Best regards,
Sarah from Accounting
sarah@company.com"""

sender = "sarah@company.com"

# Scan the email
result = await scanner.scan(safe_email, sender)
print("=== Safe Email Scan Result ===")
print(f"Risk Score: {result.risk_score}/100")
print(f"Classification: {result.classification}")
print(f"LLM Escalated: {result.llm_escalated}")
print(f"Explanation:\n{result.explanation}")

## 2. Scanning a Phishing Email

In [ ]:
# Phishing email example
phishing_email = """URGENT: Your account has been suspended!

Dear Customer,

We have detected unauthorized access to your account. You must verify your
identity immediately or your account will be permanently closed within 24 hours.

Click here to verify: http://suspicious-domain.com/verify-account?id=12345
Update your payment: http://another-suspicious-domain.xyz/payment

Please provide your bank account details and password to restore access.

Regards,
Security Team
support@fake-bank.com"""

sender = "support@fake-bank.com"

# Scan the email
result = await scanner.scan(phishing_email, sender)
print("=== Phishing Email Scan Result ===")
print(f"Risk Score: {result.risk_score}/100")
print(f"Classification: {result.classification}")
print(f"LLM Escalated: {result.llm_escalated}")
print(f"Explanation:\n{result.explanation}")

## 3. Scanning a Borderline Email (Triggers LLM Escalation)

In [ ]:
# Borderline email example (likely to get score in 45-70 range)
borderline_email = """Dear Employee,

This is a reminder about the upcoming changes to your benefits enrollment.
Please review the attached document regarding your salary adjustment and
payroll updates for the next quarter.

You may need to update your direct deposit information. Please visit
our HR portal at https://hr-portal.company.com/benefits to review.

Thank you,
HR Department
hr@company.com"""

sender = "hr@company.com"  # High-impact context (HR)

# Scan the email
result = await scanner.scan(borderline_email, sender)
print("=== Borderline Email Scan Result ===")
print(f"Risk Score: {result.risk_score}/100")
print(f"Classification: {result.classification}")
print(f"LLM Escalated: {result.llm_escalated}")
print(f"Explanation:\n{result.explanation}")

## 4. Running Evaluation Against Example Emails

In [ ]:
from src.evaluation.evaluator import evaluate, load_examples

# Load example emails
examples = load_examples('../tests/fixtures/example_emails.json')

# Run evaluation
evaluation_result = await evaluate(scanner, examples)

print("=== Evaluation Results ===")
print(f"Recall (primary):  {evaluation_result.recall:.3f}")
print(f"Precision:         {evaluation_result.precision:.3f}")
print(f"F1 Score:          {evaluation_result.f1:.3f}")
print("")
print("Per-Email Results:")
print(f"{'ID':<20} {'Predicted':<12} {'Actual':<12} {'Score':<8} {'Correct'}")
print("-" * 65)
for e in evaluation_result.per_email:
    marker = "YES" if e.correct else "** NO (FALSE NEGATIVE)" if e.actual == "Phishing" else "NO"
    print(f"{e.email_id:<20} {e.predicted:<12} {e.actual:<12} {e.risk_score:<8} {marker}")

## Key Design Decisions

1. **ML-First Approach**: Uses interpretable LogisticRegression with feature importances for explainability
2. **Conditional LLM Escalation**: Only triggers LLM for borderline scores (45-70) or high-impact contexts
3. **Non-Modifying LLM**: LLM provides enhanced explanation but never changes the ML risk score
4. **Template-Based Explanations**: Combines ML signals with optional LLM insights in plain English
5. **Recall-Prioritized Evaluation**: Focuses on minimizing false negatives (missed phishing emails)
6. **Deterministic Operation**: Fixed seeds and reproducible results for auditability